In [1]:
"""PTCG AI Battle | Variant A — Hammer-lanche Damage-Aware

Controlled candidate: same official Mega Abomasnow sample deck and same
submission packaging as Baseline 01. Only the stated attack/target scoring
changes. Run all cells to validate the policy and produce ptcg_ai_battle_variant_a_hammer_lanche.tar.gz.
This notebook does not submit to Kaggle.
"""

'PTCG AI Battle | Variant A — Hammer-lanche Damage-Aware\n\nControlled candidate: same official Mega Abomasnow sample deck and same\nsubmission packaging as Baseline 01. Only the stated attack/target scoring\nchanges. Run all cells to validate the policy and produce ptcg_ai_battle_variant_a_hammer_lanche.tar.gz.\nThis notebook does not submit to Kaggle.\n'

In [2]:
from pathlib import Path
import hashlib
import io
import os
import tempfile
import textwrap
import tarfile
import gzip
import ast
import importlib.util
import random
import sys

try:
    from IPython.display import FileLink, display
except ImportError:
    FileLink = display = None

ON_KAGGLE = Path('/kaggle/working').is_dir()
if ON_KAGGLE:
    WORKDIR = Path('/kaggle/working')
else:
    WORKDIR = Path(tempfile.mkdtemp(prefix='ptcg-ai-battle-'))
WORKDIR.mkdir(parents=True, exist_ok=True)
SOURCE_DIR = Path(tempfile.mkdtemp(prefix='ptcg-ai-battle-source-'))
# Kaggle executes the agent in an isolated folder; include the official CABT runtime.
CG_CANDIDATES = [
    Path.cwd() / 'datasets/sample_submission/sample_submission/sample_submission/cg',
    Path.cwd() / 'competitions/public-comp/ptcg-ai-battle/datasets/sample_submission/sample_submission/sample_submission/cg',
]
if Path('/kaggle/input').is_dir():
    CG_CANDIDATES.extend(Path('/kaggle/input').rglob('cg/api.py'))
CG_DIR = next((candidate for p in CG_CANDIDATES
               for candidate in [p.parent if p.is_file() and p.name == 'api.py' else p]
               if (candidate / 'api.py').is_file()), None)
if CG_DIR is None:
    raise FileNotFoundError('Official CABT cg package not found; attach the competition data.')
CG_FILES = ['__init__.py', 'api.py', 'utils.py', 'sim.py', 'libcg.so']
assert all((CG_DIR / name).is_file() for name in CG_FILES), f'Incomplete CABT package: {CG_DIR}'
print('Build directory:', WORKDIR)
print('Private source staging directory:', SOURCE_DIR)
print('Execution environment:', 'Kaggle' if ON_KAGGLE else 'local temporary directory')


Build directory: /var/folders/7k/q1n29k35429d2mnrn_3mbtjr0000gn/T/ptcg-ai-battle-q5cgm4f4
Private source staging directory: /var/folders/7k/q1n29k35429d2mnrn_3mbtjr0000gn/T/ptcg-ai-battle-source-pkgso8ei
Execution environment: local temporary directory


In [3]:
MAIN_SOURCE = '"""Variant A: estimate variable Mega Abomasnow attack damage.\n\nThis is a readable starting policy, not a claim of competitive strength.\nAll returned choices are indexes from the simulator-provided legal options.\n"""\nfrom functools import lru_cache\nfrom pathlib import Path\n\nfrom cg.api import (\n    AreaType,\n    CardType,\n    OptionType,\n    SelectContext,\n    SelectType,\n    all_attack,\n    all_card_data,\n    to_observation_class,\n)\n\n# Official Mega Abomasnow ex sample list: 25 Pokémon/trainer cards + 35 Water Energy.\nDECK = [\n    1158,\n    721, 721,\n    722, 722, 722, 722,\n    723, 723, 723, 723,\n    1145, 1145, 1145, 1145,\n    1205, 1205,\n    1227, 1227, 1227, 1227,\n    1235, 1235, 1235, 1235,\n] + [3] * 35\n\n\ndef _int(value, default=-1):\n    try:\n        return int(value)\n    except (TypeError, ValueError):\n        return default\n\n\n@lru_cache(maxsize=1)\ndef _cards():\n    return {card.cardId: card for card in all_card_data()}\n\n\n@lru_cache(maxsize=1)\ndef _attacks():\n    return {attack.attackId: attack for attack in all_attack()}\n\n\ndef _player(state, player_index=None):\n    if state is None:\n        return None\n    index = state.yourIndex if player_index is None else player_index\n    if index is None or not (0 <= index < len(state.players)):\n        return None\n    return state.players[index]\n\n\ndef _pokemon(state, player_index, area, index):\n    player = _player(state, player_index)\n    if player is None or index is None:\n        return None\n    area = _int(area)\n    if area == int(AreaType.ACTIVE):\n        return player.active[index] if 0 <= index < len(player.active) else None\n    if area == int(AreaType.BENCH):\n        return player.bench[index] if 0 <= index < len(player.bench) else None\n    return None\n\n\ndef _card_from_option(obs, option):\n    state = obs.current\n    area = _int(getattr(option, \'area\', None))\n    index = getattr(option, \'index\', None)\n    player_index = getattr(option, \'playerIndex\', None)\n    # PLAY is documented with a hand index and may omit area=HAND.\n    if _int(getattr(option, \'type\', None)) == int(OptionType.PLAY):\n        player = _player(state, player_index)\n        hand = getattr(player, \'hand\', None) if player else None\n        if hand is not None and index is not None and 0 <= index < len(hand):\n            return hand[index]\n    if area == int(AreaType.HAND):\n        player = _player(state, player_index)\n        hand = getattr(player, \'hand\', None) if player else None\n        if hand is not None and index is not None and 0 <= index < len(hand):\n            return hand[index]\n    if area == int(AreaType.DECK) and obs.select and obs.select.deck:\n        deck = obs.select.deck\n        if index is not None and 0 <= index < len(deck):\n            return deck[index]\n    if area == int(AreaType.LOOKING) and state and state.looking:\n        looking = state.looking\n        if index is not None and 0 <= index < len(looking):\n            return looking[index]\n    card_id = getattr(option, \'cardId\', None)\n    if card_id is not None:\n        return type(\'CardRef\', (), {\'id\': card_id})()\n    return None\n\n\ndef _card_priority(card_id):\n    # Deck-specific priorities: Mega Abomasnow, Snover, and Kyogre.\n    return {723: 100, 722: 70, 721: 50}.get(card_id, 10)\n\n\ndef _basic_pokemon(card):\n    if card is None:\n        return False\n    data = _cards().get(card.id)\n    return bool(data and data.cardType == CardType.POKEMON and data.basic)\n\n\ndef _target_score(state, player_index, area, index):\n    target = _pokemon(state, player_index, area, index)\n    if target is None:\n        return 0\n    score = _card_priority(target.id)\n    # Prefer energizing the main line, especially while it approaches its two-energy attack.\n    energies = len(getattr(target, \'energies\', []) or [])\n    if target.id == 723:\n        score += max(0, 3 - energies) * 18\n    elif target.id == 722:\n        score += 12 if energies == 0 else 0\n    elif target.id == 721:\n        score += max(0, 2 - energies) * 8\n    if _int(area) == int(AreaType.ACTIVE):\n        score += 5\n    return score\n\n\ndef _expected_hammer_damage(obs):\n    """Estimate Hammer-lanche damage from the still-hidden card pool."""\n    state = obs.current\n    me = _player(state)\n    if me is None:\n        return 0.0\n\n    known_cards = list(me.hand or []) + list(me.discard or [])\n    prizes = list(me.prize or [])\n    known_cards.extend(card for card in prizes if card is not None)\n    known_cards.extend(card for card in (state.looking or []) if card is not None)\n    for pokemon in list(me.active or []) + list(me.bench or []):\n        if pokemon is not None:\n            known_cards.extend(getattr(pokemon, \'energyCards\', []) or [])\n\n    known_water = sum(1 for card in known_cards if getattr(card, \'id\', None) == 3)\n    hidden_prizes = sum(1 for card in prizes if card is None)\n    hidden_pool = max(1, int(me.deckCount) + hidden_prizes)\n    remaining_water = max(0, DECK.count(3) - known_water)\n    top_count = min(6, max(0, int(me.deckCount)))\n    expected_hits = top_count * remaining_water / hidden_pool\n    return 100.0 * expected_hits\n\ndef _main_score(obs, option):\n    kind = _int(option.type)\n    state = obs.current\n    me = _player(state)\n\n    if kind == int(OptionType.END):\n        return 0\n    if kind == int(OptionType.EVOLVE):\n        card = _card_from_option(obs, option)\n        return 600 + _card_priority(getattr(card, \'id\', -1))\n    if kind == int(OptionType.PLAY):\n        card = _card_from_option(obs, option)\n        card_id = getattr(card, \'id\', -1)\n        if card_id == 1235:  # Waitress: attach a Basic Energy from the top six.\n            return 520\n        if card_id == 1145:  # Mega Signal: find the main evolution.\n            return 510\n        if card_id == 1205:  # Cyrano: search Pokémon ex.\n            return 490\n        if card_id == 1227:  # Lillie\'s Determination: redraw when hand is small.\n            return 470 if me and me.handCount <= 5 else 300\n        if _basic_pokemon(card):\n            occupied = len([pokemon for pokemon in (me.active or []) if pokemon]) + len(me.bench)\n            return 420 if me and occupied < me.benchMax else 50\n        data = _cards().get(card_id)\n        if data and data.cardType in (CardType.ITEM, CardType.SUPPORTER, CardType.STADIUM):\n            return 280\n        if data and data.cardType == CardType.TOOL:\n            return 180\n        return 100\n    if kind == int(OptionType.ABILITY):\n        return 440\n    if kind == int(OptionType.ATTACH):\n        return 380 + _target_score(state, state.yourIndex, option.inPlayArea, option.inPlayIndex)\n    if kind == int(OptionType.ATTACK):\n        attack = _attacks().get(getattr(option, \'attackId\', None))\n        attack_id = getattr(option, \'attackId\', None)\n        if attack_id == 1046:  # Hammer-lanche: 100 per Water Energy in the top six.\n            return 200 + _expected_hammer_damage(obs) / 100\n        return 200 + (getattr(attack, \'damage\', 0) or 0) / 100\n    if kind == int(OptionType.RETREAT):\n        active = (me.active[0] if me and me.active else None)\n        if active and (me.asleep or me.paralyzed):\n            return 410\n        if active and active.hp <= max(30, active.maxHp // 4) and me.bench:\n            return 260\n        return 20\n    return 10\n\n\ndef _selection_score(obs, option):\n    select = obs.select\n    context = _int(select.context)\n    kind = _int(option.type)\n    state = obs.current\n    me = _player(state)\n    card = _card_from_option(obs, option)\n    card_id = getattr(card, \'id\', -1)\n\n    if kind == int(OptionType.YES):\n        if context == int(SelectContext.IS_FIRST):\n            return 0  # Choose to go second with the fast Water-energy deck.\n        if context == int(SelectContext.MULLIGAN):\n            hand = getattr(me, \'hand\', None) if me else None\n            return 100 if hand is not None and not any(_basic_pokemon(item) for item in hand) else 0\n        return 100\n    if kind == int(OptionType.NO):\n        return 1 if context == int(SelectContext.IS_FIRST) else 0\n    if kind == int(OptionType.NUMBER):\n        return getattr(option, \'number\', 0) or 0\n    if kind == int(OptionType.SPECIAL_CONDITION):\n        # Prefer paralysis, then poison, burn, sleep, confusion.\n        return {3: 5, 0: 4, 1: 3, 2: 2, 4: 1}.get(_int(option.specialConditionType), 0)\n\n    if context == int(SelectContext.SETUP_ACTIVE_POKEMON):\n        return {721: 200, 722: 150}.get(card_id, _card_priority(card_id))\n    if context in (int(SelectContext.SETUP_BENCH_POKEMON), int(SelectContext.TO_FIELD), int(SelectContext.TO_BENCH)):\n        return {722: 220, 721: 180, 723: 120}.get(card_id, _card_priority(card_id))\n    if context in (int(SelectContext.TO_HAND), int(SelectContext.LOOK)):\n        return {723: 300, 722: 220, 721: 180, 1235: 100}.get(card_id, 50)\n    if context in (int(SelectContext.ATTACH_TO), int(SelectContext.EVOLVES_FROM), int(SelectContext.EVOLVE)):\n        return _target_score(state, option.playerIndex, option.area, option.index)\n    if context in (int(SelectContext.DAMAGE), int(SelectContext.DAMAGE_COUNTER), int(SelectContext.DAMAGE_COUNTER_ANY)):\n        target = _pokemon(state, option.playerIndex, option.area, option.index)\n        if target is None:\n            return -1000\n        is_own = state is not None and option.playerIndex == state.yourIndex\n        if context == int(SelectContext.DAMAGE) and is_own:\n            return -1000\n        return 1000 / max(1, target.hp) + (100 if target.hp <= 100 else 0)\n    if context in (int(SelectContext.HEAL), int(SelectContext.REMOVE_DAMAGE_COUNTER)):\n        target = _pokemon(state, option.playerIndex, option.area, option.index)\n        return target.maxHp - target.hp if target else 0\n    if context in (int(SelectContext.DISCARD), int(SelectContext.DISCARD_ENERGY_CARD),\n                   int(SelectContext.DISCARD_TOOL_CARD), int(SelectContext.DISCARD_CARD_OR_ATTACHED_CARD)):\n        data = _cards().get(card_id)\n        if data and data.cardType == CardType.POKEMON:\n            return -100 - _card_priority(card_id)\n        if data and data.cardType in (CardType.BASIC_ENERGY, CardType.SPECIAL_ENERGY):\n            return -30\n        return -_card_priority(card_id)\n    if context in (int(SelectContext.TO_DECK), int(SelectContext.TO_DECK_BOTTOM)):\n        return -_card_priority(card_id)\n    if context == int(SelectContext.SKILL_ORDER):\n        return 1\n    return _card_priority(card_id)\n\n\ndef _choose(obs):\n    select = obs.select\n    options = list(select.option or [])\n    if not options:\n        return []\n    if _int(select.type) == int(SelectType.MAIN):\n        return [max(range(len(options)), key=lambda index: (_main_score(obs, options[index]), -index))]\n\n    ranked = sorted(range(len(options)), key=lambda index: (_selection_score(obs, options[index]), -index), reverse=True)\n    context = _int(select.context)\n    discard_contexts = {\n        int(SelectContext.DISCARD), int(SelectContext.DISCARD_ENERGY_CARD),\n        int(SelectContext.DISCARD_TOOL_CARD), int(SelectContext.DISCARD_CARD_OR_ATTACHED_CARD),\n    }\n    requested = select.minCount if context in discard_contexts else select.maxCount\n    count = max(select.minCount, min(requested, select.maxCount, len(ranked)))\n    return ranked[:count]\n\n\ndef agent(obs_dict):\n    """Return a legal option-index list, or the fixed 60-card deck at startup."""\n    obs = to_observation_class(obs_dict)\n    if obs.select is None:\n        return list(DECK)\n    return _choose(obs)\n'
DECK_SOURCE = '1158\n721\n721\n722\n722\n722\n722\n723\n723\n723\n723\n1145\n1145\n1145\n1145\n1205\n1205\n1227\n1227\n1227\n1227\n1235\n1235\n1235\n1235\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n'

MAIN_PATH = SOURCE_DIR / 'main.py'
DECK_PATH = SOURCE_DIR / 'deck.csv'
MAIN_PATH.write_text(MAIN_SOURCE, encoding='utf-8', newline='\n')
DECK_PATH.write_text(DECK_SOURCE, encoding='ascii', newline='\n')

# Static and structural checks before importing any optional simulator package.
source_bytes = MAIN_PATH.read_bytes()
compile(source_bytes, str(MAIN_PATH), 'exec')
ast.parse(source_bytes, filename=str(MAIN_PATH))
deck_ids = [int(line) for line in DECK_PATH.read_text(encoding='ascii').splitlines() if line.strip()]
assert len(deck_ids) == 60, f'Expected exactly 60 cards; found {len(deck_ids)}'
assert deck_ids == [1158, 721, 721, 722, 722, 722, 722, 723, 723, 723, 723,
                    1145, 1145, 1145, 1145, 1205, 1205, 1227, 1227, 1227, 1227,
                    1235, 1235, 1235, 1235] + [3] * 35
print('main.py bytes:', len(source_bytes))
print('main.py SHA-256:', hashlib.sha256(source_bytes).hexdigest())
print('Deck size:', len(deck_ids), '| unique card IDs:', len(set(deck_ids)))


main.py bytes: 11422
main.py SHA-256: 7ef1b495c35704f18edaa7c10c7d9d5b28b1131ff9af5c0ca95342b149469bff
Deck size: 60 | unique card IDs: 9


In [4]:
# Verify that the bundled CABT Python wrapper is importable and can initialize the agent.
PROJECT_ROOT = Path.cwd()
possible_cg_dirs = [
    PROJECT_ROOT / 'datasets/sample_submission/sample_submission/sample_submission/cg',
    PROJECT_ROOT / 'competitions/public-comp/ptcg-ai-battle/datasets/sample_submission/sample_submission/sample_submission/cg',
]
if Path('/kaggle/input').is_dir():
    possible_cg_dirs += [
        Path('/kaggle/input/the-pokemon-company-ptcg-ai-battle-challenge-playground/sample_submission/sample_submission/cg'),
        Path('/kaggle/input/ptcg-ai-battle-sample-submission/sample_submission/cg'),
    ]
CG_DIR = next((path for path in possible_cg_dirs if (path / 'api.py').is_file()), CG_DIR)
if CG_DIR is not None:
    sys.path.insert(0, str(CG_DIR.parent))
    spec = importlib.util.spec_from_file_location('ptcg_baseline_01', MAIN_PATH)
    policy = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(policy)
    assert callable(policy.agent)
    assert len(policy.agent({'select': None, 'logs': [], 'current': None})) == 60
    from types import SimpleNamespace
    waitress = SimpleNamespace(id=1235)
    play_option = SimpleNamespace(type=policy.OptionType.PLAY, area=None, index=0, playerIndex=None,
                                 cardId=None, inPlayArea=None, inPlayIndex=None)
    fake_state = SimpleNamespace(yourIndex=0, players=[SimpleNamespace(hand=[waitress], active=[],
                                 bench=[], benchMax=5, handCount=1)])
    fake_observation = SimpleNamespace(current=fake_state, select=None)
    assert policy._card_from_option(fake_observation, play_option).id == 1235
    assert policy._main_score(fake_observation, play_option) == 520
    print('CABT wrapper:', CG_DIR)
    fake_me = SimpleNamespace(deckCount=54, hand=[], discard=[], prize=[None] * 6, active=[], bench=[])
    fake_state = SimpleNamespace(yourIndex=0, players=[fake_me], looking=[])
    fake_obs = SimpleNamespace(current=fake_state)
    assert abs(policy._expected_hammer_damage(fake_obs) - 350.0) < 1e-9
    print('Variant A check: 35/60 Water Energy implies 350 expected Hammer-lanche damage.')
    print('Agent imports, returns the exact deck, resolves PLAY options, and passes the variant-specific scoring check.')


CABT wrapper: /Users/flexonafft/MLKaggleTasks/competitions/public-comp/ptcg-ai-battle/datasets/sample_submission/sample_submission/sample_submission/cg
Variant A check: 35/60 Water Energy implies 350 expected Hammer-lanche damage.
Agent imports, returns the exact deck, resolves PLAY options, and passes the variant-specific scoring check.


In [5]:
# Run ten complete local games, alternating which seat uses our policy.
# This is only an interface/smoke check against a random-choice opponent.
if CG_DIR is not None:
    from cg.game import battle_finish, battle_select, battle_start

    def random_legal_agent(observation):
        select = observation.get('select')
        if select is None:
            return list(deck_ids)
        options = select.get('option') or []
        minimum = int(select.get('minCount', 0))
        maximum = int(select.get('maxCount', len(options)))
        count = random.randint(minimum, min(maximum, len(options)))
        return random.sample(range(len(options)), count)

    def validate_choice(observation, choice):
        select = observation['select']
        options = select.get('option') or []
        assert isinstance(choice, list) and all(isinstance(i, int) for i in choice)
        assert len(choice) == len(set(choice)), 'Duplicate option indexes'
        assert all(0 <= i < len(options) for i in choice), 'Out-of-range option index'
        assert int(select.get('minCount', 0)) <= len(choice) <= int(select.get('maxCount', len(options)))

    def play_smoke_game(seed, policy_seat):
        random.seed(seed)
        observation, start = battle_start(deck_ids, deck_ids, reverse_player=False)
        if observation is None:
            raise RuntimeError(f'Could not start game: {start.errorPlayer=} {start.errorType=}')
        try:
            for decision in range(10000):
                state = observation.get('current') or {}
                result = int(state.get('result', -1))
                if result != -1:
                    return result, decision
                select = observation.get('select')
                if select is None:
                    raise RuntimeError('Live game returned no legal selection.')
                player_index = int(state.get('yourIndex', 0))
                if player_index == policy_seat:
                    choice = policy.agent(observation)
                else:
                    choice = random_legal_agent(observation)
                validate_choice(observation, choice)
                observation = battle_select(choice)
            raise RuntimeError('Game exceeded 10,000 decisions.')
        finally:
            battle_finish()

    smoke_results = []
    for game_number, seat in enumerate((0, 1, 0, 1, 0, 1, 0, 1, 0, 1), start=1):
        result, decisions = play_smoke_game(20261003 + game_number, seat)
        won = result == seat
        smoke_results.append({'game': game_number, 'policy_seat': seat, 'winner': result,
                              'policy_win': won, 'decisions': decisions})
        print(f"game={game_number} policy_seat={seat} winner={result} policy_win={won} decisions={decisions}")
    print('Smoke games completed (10 games, alternating seats). This tiny random-opponent sample is not a strength estimate.')


game=1 policy_seat=0 winner=0 policy_win=True decisions=28
game=2 policy_seat=1 winner=1 policy_win=True decisions=28
game=3 policy_seat=0 winner=0 policy_win=True decisions=30
game=4 policy_seat=1 winner=1 policy_win=True decisions=24
game=5 policy_seat=0 winner=0 policy_win=True decisions=78
game=6 policy_seat=1 winner=1 policy_win=True decisions=23
game=7 policy_seat=0 winner=0 policy_win=True decisions=100
game=8 policy_seat=1 winner=1 policy_win=True decisions=22
game=9 policy_seat=0 winner=0 policy_win=True decisions=20
game=10 policy_seat=1 winner=1 policy_win=True decisions=27
Smoke games completed (10 games, alternating seats). This tiny random-opponent sample is not a strength estimate.


In [6]:
# Build a deterministic archive with exactly the files expected at archive root.
ARCHIVE = WORKDIR / 'ptcg_ai_battle_variant_a_hammer_lanche.tar.gz'
with ARCHIVE.open('wb') as raw_file:
    with gzip.GzipFile(filename='', mode='wb', fileobj=raw_file, mtime=0) as zipped:
        with tarfile.open(fileobj=zipped, mode='w', format=tarfile.GNU_FORMAT) as archive:
            archive_files = [('main.py', MAIN_PATH), ('deck.csv', DECK_PATH)] + [
                ('cg/' + name, CG_DIR / name) for name in CG_FILES
            ]
            for name, path in archive_files:
                payload = path.read_bytes()
                info = tarfile.TarInfo(name)
                info.size = len(payload)
                info.mode = 0o644
                info.mtime = 0
                info.uid = info.gid = 0
                info.uname = info.gname = ''
                archive.addfile(info, io.BytesIO(payload))

with tarfile.open(ARCHIVE, mode='r:gz') as archive:
    assert archive.getnames() == ['main.py', 'deck.csv', 'cg/__init__.py', 'cg/api.py',
                                  'cg/utils.py', 'cg/sim.py', 'cg/libcg.so']
    assert archive.extractfile('main.py').read() == MAIN_PATH.read_bytes()
    assert archive.extractfile('deck.csv').read() == DECK_PATH.read_bytes()
assert ARCHIVE.stat().st_size < 197.7 * 1024 * 1024
print('Bundle:', ARCHIVE)
print('Bundle bytes:', ARCHIVE.stat().st_size)
print('Bundle SHA-256:', hashlib.sha256(ARCHIVE.read_bytes()).hexdigest())
if FileLink and ON_KAGGLE:
    display(FileLink(str(ARCHIVE)))


Bundle: /var/folders/7k/q1n29k35429d2mnrn_3mbtjr0000gn/T/ptcg-ai-battle-q5cgm4f4/ptcg_ai_battle_variant_a_hammer_lanche.tar.gz
Bundle bytes: 532804
Bundle SHA-256: 12b1aaf8907676cbe9cb35dd6abdee348bba85e426f679493975a5c1d20b9fd8


In [7]:
assert ARCHIVE.is_file() and ARCHIVE.suffixes[-2:] == ['.tar', '.gz']
print('Select this Kaggle output file:')
print(ARCHIVE.name)
print('Submission entry point: agent(obs_dict) -> list[int]')
print('Inside the archive: main.py, deck.csv, cg Python modules, and libcg.so')
print('Candidate: Variant A — Hammer-lanche Damage-Aware')
print('No Kaggle submission was sent.')
if FileLink and ON_KAGGLE:
    display(FileLink(str(ARCHIVE)))


Select this Kaggle output file:
ptcg_ai_battle_variant_a_hammer_lanche.tar.gz
Submission entry point: agent(obs_dict) -> list[int]
Inside the archive: main.py, deck.csv, cg Python modules, and libcg.so
Candidate: Variant A — Hammer-lanche Damage-Aware
No Kaggle submission was sent.
